<a href="https://colab.research.google.com/github/iam6ar0r/BASCV/blob/main/Fake_randomForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
df = pd.read_csv('WELFake_Dataset.csv')

# Combine title and text
df['clean_text'] = df['title'].fillna('') + " " + df['text'].fillna('')

df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

# Separating for info
df_fake = df[df['label'] == 0]
df_true = df[df['label'] == 1]

print(f"Total fake news samples: {len(df_fake)}")
print(f"Total real news samples: {len(df_true)}")

Total fake news samples: 35028
Total real news samples: 37106


# **seperating**

In [3]:
X = df['clean_text'].astype('str')
y = df['label']

# seperating into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Data splitting complete. Ready for vectorization.")

# Convert words to numbers using TF-IDF
vectorizer = TfidfVectorizer(stop_words='english', max_df=0.85, min_df=2, max_features=50000)

# Fit on training data only, then transform both sets
X_train = vectorizer.fit_transform(X_train)
X_test  = vectorizer.transform(X_test)

print("Vectorization complete!")
print("Training set size:", X_train.shape)

Data splitting complete. Ready for vectorization.
Vectorization complete!
Training set size: (57707, 50000)


# **training and testing**

In [5]:
# Train the model
model = RandomForestClassifier(n_estimators=50, random_state=42, class_weight='balanced', n_jobs=-1)
model.fit(X_train, y_train)

# Evaluate on test set
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Model accuracy: {accuracy * 100:.2f}%")
print("\nModel Classification Report:")
print(classification_report(y_test, predictions))

Model accuracy: 94.68%

Model Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.94      0.95      7089
           1       0.94      0.96      0.95      7338

    accuracy                           0.95     14427
   macro avg       0.95      0.95      0.95     14427
weighted avg       0.95      0.95      0.95     14427



# **interface**

In [10]:
import ipywidgets as widgets
from IPython.display import display

text_input = widgets.Text(
    value='',
    placeholder='Enter the news article here...',
    description='News:',
    layout=widgets.Layout(width='80%')
)

button = widgets.Button(
    description='Analyze News',
    button_style='info'
)

output = widgets.Output()

def on_button_click(b):
    with output:
        output.clear_output()

        input_data = vectorizer.transform([text_input.value])

        prediction = model.predict(input_data)
        proba = model.predict_proba(input_data)[0]

        real_pct = round(proba[0] * 100, 1)
        fake_pct = round(proba[1] * 100, 1)

        if prediction[0] == 0:
            print(f"Result: ✅ REAL NEWS")
            print(f"Confidence: Real {real_pct}% | Fake {fake_pct}%")
        else:
            print(f"Result: ❌ FAKE NEWS")
            print(f"Confidence: Fake {fake_pct}% | Real {real_pct}%")

button.on_click(on_button_click)

display(text_input, button, output)

Text(value='', description='News:', layout=Layout(width='80%'), placeholder='Enter the news article here...')

Button(button_style='info', description='Analyze News', style=ButtonStyle())

Output()